# Fine-tuning FLAN-T5-small for LinkedIn Job Description Generation

**Goal.** Take structured job inputs (title, skills, experience level, etc.) and produce a natural-language job description. We do this by *fine-tuning* a small pretrained text-to-text model (FLAN-T5-small, ~77M parameters) on a cleaned subset of LinkedIn postings.

**Why FLAN-T5-small?** It is a sequence-to-sequence (encoder–decoder) Transformer that has already been pretrained on a large mixture of language tasks. "Small" means it actually fits in 6 GB of VRAM with mixed precision. "FLAN" means it has been instruction-tuned, so it already understands prompts like *"generate a job description for the following role."*.

**Plan of the notebook.**
1. Setup: install/import everything, set paths.
2. Load and clean data.
3. Build prompts and targets, split into train/val/test (80/10/10).
4. Tokenize and wrap in a PyTorch `Dataset` / `DataLoader`.
5. Zero-shot baseline: what does FLAN-T5-small produce with no fine-tuning?
6. Fine-tune FLAN-T5-small (manual loop, AMP/FP16).
7. Train the same architecture *from scratch* (random init) for comparison.
8. Generate samples from all three models and compare against the ground truth.

After Section 6 succeeds, the trained checkpoint in `./checkpoint/` is consumed by `app.py` (the local web UI).

Run each section and check the output before moving on.

## Section 1 — Setup

We need the following libraries:

- **`torch`** — you already know this. PyTorch tensors, autograd, training loop.
- **`transformers`** — HuggingFace's library. It gives us the FLAN-T5 model class (`T5ForConditionalGeneration`), the matching tokenizer (`AutoTokenizer`), and a learning-rate scheduler. *New concept:* HuggingFace ships pretrained model weights and the code that runs them, so `from_pretrained("google/flan-t5-small")` downloads the weights and builds the `nn.Module` for you. We will see this for the first time in Section 5 (zero-shot baseline).
- **`sentencepiece`** — the underlying tokenizer algorithm T5 uses (subword units). You don't call it directly; `AutoTokenizer` does.
- **`pandas`** — CSV loading and joins.
- **`langdetect`** — language identification. We will use it to keep only English postings.
- **`tqdm`** — progress bars.

**Run the cell below once.** If you already have these installed, pip will simply skip them. The `-q` flag keeps the output quiet.

In [13]:
# One-time install. Comment out after the first successful run if you want.
%pip install -q torch transformers sentencepiece pandas langdetect tqdm

Note: you may need to restart the kernel to use updated packages.


c:\Users\AlessandroMacchi\Downloads\Progetti\linkedin-jobpostings--writer\.venv\Scripts\python.exe: No module named pip


In [14]:
# Standard library
import os
import re
import random
from pathlib import Path

# Numerical / data
import numpy as np
import pandas as pd

# PyTorch
import torch
from torch.utils.data import Dataset, DataLoader

# HuggingFace transformers — we import only what we will use later in the notebook.
# AutoTokenizer  : picks the right tokenizer class for a given model name.
# T5ForConditionalGeneration : the seq2seq model class we will fine-tune.
# T5Config       : the architecture config (used in Section 7 for random init).
# get_linear_schedule_with_warmup : LR scheduler (linear warmup, then linear decay).
from transformers import (
    AutoTokenizer,
    T5ForConditionalGeneration,
    T5Config,
    get_linear_schedule_with_warmup,
)

# Language detection (used in Section 2 to filter English-only postings)
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0  # makes langdetect deterministic across runs

# Progress bars
from tqdm.auto import tqdm
tqdm.pandas()  # enables df.progress_apply(...) for the slow language-detection step

print("Imports OK")

Imports OK


In [15]:
# Reproducibility: fix every RNG we touch. With seed=42 the train/val/test split,
# the 200k subsample, and the optimizer's stochastic behavior will all be repeatable.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Paths. Everything lives next to the notebook.
DATA_DIR = Path("./data")                              # the 11 CSVs you already have
CHECKPOINT_DIR = Path("./checkpoint")                  # fine-tuned FLAN-T5-small (Section 6)
CHECKPOINT_SCRATCH_DIR = Path("./checkpoint_scratch")  # from-scratch comparison model (Section 7)
CHECKPOINT_DIR.mkdir(exist_ok=True)
CHECKPOINT_SCRATCH_DIR.mkdir(exist_ok=True)

# Device. The GTX 1660 Super has no Tensor Cores, but FP16 still saves memory,
# which is the binding constraint at 6 GB VRAM. We will enable AMP in Section 6.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Model name. We refer to it by its HuggingFace Hub ID; downloading happens later
# in Section 4 (tokenizer) and Section 5 (weights), the first time it is requested.
MODEL_NAME = "google/flan-t5-small"
print(f"Model: {MODEL_NAME}")

# Sanity check: the data directory should contain the CSVs we will load next.
csvs = sorted(p.name for p in DATA_DIR.glob("*.csv"))
print(f"\nFound {len(csvs)} CSVs in {DATA_DIR}/:")
for name in csvs:
    print(f"  {name}")

Device: cpu
Model: google/flan-t5-small

Found 11 CSVs in data/:
  benefits.csv
  companies.csv
  company_industries.csv
  company_specialities.csv
  employee_counts.csv
  industries.csv
  job_industries.csv
  job_skills.csv
  postings.csv
  salaries.csv
  skills.csv


**What you should see when you run the cells above:**

1. The `%pip install` cell finishes without errors (may say "already satisfied" for most packages).
2. `Imports OK`.
3. `Device: cuda`, the GPU name (`NVIDIA GeForce GTX 1660 SUPER`), and ~6 GB VRAM. If it says `cpu`, your CUDA-enabled PyTorch is not installed correctly — fix that before going further, otherwise training will be impractically slow.
4. A list of the 11 CSV files in `./data/`.

**Stop here.** Run all three cells in this section, confirm the output looks right, and tell me to continue with Section 2.

## Section 2 — Load and clean data

We have 11 CSVs but most aren't needed. We pull only the columns we use, then:

1. Drop rows missing the `description` or `title` (we can't train without them).
2. Keep only postings whose description has between **50 and 700 words**. Too short → not enough signal; too long → wastes our 384-token target budget later.
3. Keep only **English** postings, by running `langdetect` on the first 500 characters of each description. *This step is slow — expect ~5 minutes.*
4. Join the human-readable **skill names** (`job_skills.csv` + `skills.csv`) and **industry names** (`job_industries.csv` + `industries.csv`) onto each job.

Missing-value behavior we will rely on later in Section 3:

- `formatted_experience_level` is missing in ~24% of rows.

We don't fix this here — we leave the `NaN`s and translate them to `"not specified"` inside `build_prompt(...)` in Section 3.

In [16]:
# Columns we need from postings.csv. Loading only these keeps memory in check --
# postings.csv has ~30 columns and >100k rows.
POSTINGS_COLS = [
    "job_id",
    "title",
    "description",
    "location",
    "formatted_work_type",
    "formatted_experience_level",
]

postings = pd.read_csv(DATA_DIR / "postings.csv", usecols=POSTINGS_COLS)
print(f"Loaded postings.csv: {len(postings):,} rows, {postings.shape[1]} columns")
print("\nMissing-value % per column:")
print((postings.isna().mean() * 100).round(1).sort_values(ascending=False))
postings.head(2)

Loaded postings.csv: 123,849 rows, 6 columns

Missing-value % per column:
formatted_experience_level    23.7
job_id                         0.0
title                          0.0
description                    0.0
location                       0.0
formatted_work_type            0.0
dtype: float64


,job_id,title,description,location,formatted_work_type,formatted_experience_level
0,921716,Marketing Coordinator,Job descriptionA leading real estate firm in N...,"Princeton, NJ",Full-time,NaN
1,1829192,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...","Fort Collins, CO",Full-time,NaN


In [17]:
# Step 1: drop rows missing title or description.
before = len(postings)
postings = postings.dropna(subset=["title", "description"]).reset_index(drop=True)
print(f"After dropping missing title/description: {len(postings):,} rows  ({before - len(postings):,} dropped)")

# Step 2: keep only descriptions between 50 and 700 words.
# Plain whitespace split is good enough for a length filter -- we're not measuring
# tokens here, just deciding what's reasonable for a job description.
word_counts = postings["description"].str.split().str.len()
before = len(postings)
postings = postings[(word_counts >= 50) & (word_counts <= 700)].reset_index(drop=True)
print(f"After 50-700 word filter:                {len(postings):,} rows  ({before - len(postings):,} dropped)")

After dropping missing title/description: 123,842 rows  (7 dropped)
After 50-700 word filter:                92,130 rows  (31,712 dropped)


In [18]:
# Step 3: keep only English postings.
# langdetect raises LangDetectException on empty/garbled text -- wrap in try/except.
# We classify on the first 500 chars to keep it fast; full descriptions don't
# change the answer in practice.
def is_english(text: str) -> bool:
    try:
        return detect(str(text)[:500]) == "en"
    except Exception:
        return False

print("Detecting language on each description (this is the slow step, ~5 minutes)...")
mask_en = postings["description"].progress_apply(is_english)
before = len(postings)
postings = postings[mask_en].reset_index(drop=True)
print(f"After English-only filter: {len(postings):,} rows  ({before - len(postings):,} dropped)")

Detecting language on each description (this is the slow step, ~5 minutes)...


  0%|          | 0/92130 [00:00<?, ?it/s]

After English-only filter: 91,971 rows  (159 dropped)


### Joining skill names and industry names

`postings.csv` only has `job_id`. The human-readable skill and industry **names** live in separate tables that we reach through bridge tables:

- `job_skills.csv (job_id, skill_abr)` × `skills.csv (skill_abr, skill_name)` → list of skill names per job.
- `job_industries.csv (job_id, industry_id)` × `industries.csv (industry_id, industry_name)` → list of industry names per job.

We aggregate each list into a single comma-separated string per `job_id` so it slots cleanly into our prompt template later. Sorted + deduplicated so the strings are stable across runs.

In [19]:
# Skills: bridge job_id <-> skill_abr <-> skill_name.
job_skills = pd.read_csv(DATA_DIR / "job_skills.csv")
skills     = pd.read_csv(DATA_DIR / "skills.csv")

skills_joined = (
    job_skills.merge(skills, on="skill_abr", how="left")
              .dropna(subset=["skill_name"])
              .groupby("job_id")["skill_name"]
              .apply(lambda s: ", ".join(sorted(set(s))))
              .rename("skills_joined")
              .reset_index()
)
print(f"Skill rows: {len(job_skills):,}  ->  unique jobs with skills: {len(skills_joined):,}")

# Industries: bridge job_id <-> industry_id <-> industry_name.
job_industries = pd.read_csv(DATA_DIR / "job_industries.csv")
industries     = pd.read_csv(DATA_DIR / "industries.csv")

industries_joined = (
    job_industries.merge(industries, on="industry_id", how="left")
                  .dropna(subset=["industry_name"])
                  .groupby("job_id")["industry_name"]
                  .apply(lambda s: ", ".join(sorted(set(s))))
                  .rename("industries_joined")
                  .reset_index()
)
print(f"Industry rows: {len(job_industries):,}  ->  unique jobs with industries: {len(industries_joined):,}")

# Left-join onto our filtered postings so a job with no skills/industries is kept
# (those columns become NaN and later get rewritten as "not specified").
df = (
    postings
    .merge(skills_joined,     on="job_id", how="left")
    .merge(industries_joined, on="job_id", how="left")
)
print(f"\nFinal cleaned dataframe: {len(df):,} rows, {df.shape[1]} columns")
print(df.columns.tolist())
df.tail()

Skill rows: 213,768  ->  unique jobs with skills: 126,807
Industry rows: 164,808  ->  unique jobs with industries: 127,025

Final cleaned dataframe: 91,971 rows, 8 columns
['job_id', 'title', 'description', 'location', 'formatted_work_type', 'formatted_experience_level', 'skills_joined', 'industries_joined']


,job_id,title,description,location,formatted_work_type,formatted_experience_level,skills_joined,industries_joined
91966,3906266212,Phlebotomist - Float,Job Description\n\nThe Patient Services Repres...,"Carroll County, MD",Contract,Entry level,Science,Staffing and Recruiting
91967,3906266272,Quality Engineer,Position: Quality Engineer I (Complaint Invest...,"Irvine, CA",Contract,Mid-Senior level,Engineering,"Biotechnology Research, Medical Equipment Manu..."
91968,3906267117,Title IX/Investigations Attorney,Our Walnut Creek office is currently seeking a...,"Walnut Creek, CA",Full-time,Mid-Senior level,"Business Development, Legal",Law Practice
91969,3906267195,Business Development Manager,The Business Development Manager is a 'hunter'...,"Texas, United States",Full-time,NaN,"Business Development, Sales",Industrial Machinery Manufacturing
91970,3906267224,Marketing Social Media Specialist,Marketing Social Media Specialist - $70k – $75...,"San Juan Capistrano, CA",Full-time,Mid-Senior level,Marketing,Manufacturing


**What you should see:**

- Row counts shrinking after each filter (drop missing → word-count → English).
- After the English filter, roughly **80–100k** rows (exact number depends on your copy of the data).
- `df` has the original posting fields plus two new columns: `skills_joined` and `industries_joined`.
- `df.head(2)` shows real-looking postings with comma-separated skill/industry strings.
- Some rows will have `NaN` in `normalized_salary`, `formatted_experience_level`, `skills_joined`, or `industries_joined` — that is expected and gets handled in Section 3.

**Stop here.** Run the cells, watch the row counts go down, and tell me to continue with Section 3.

## Section 3 — Build prompts and targets

We turn each row of `df` into a (prompt, target) pair for sequence-to-sequence training.

**Why the prompt format matters.** The fine-tuned model learns to map *exactly this prompt template* to a job description. If the inference-time prompt (in `app.py`) differs by even punctuation or label order, output quality silently degrades. We define `build_prompt(row)` once here and **paste the same function verbatim into `app.py`**. Any divergence is a bug.

**Missing values.** Every NaN field is rewritten as `"not specified"`.

**Target.** The cleaned description with whitespace collapsed (multiple spaces/newlines become one space).

**Subsample + split.** Random subsample to 200k examples (or all rows if fewer), 80/10/10 train/val/test split with `seed=42`. We keep them in memory as DataFrames and don't write to disk.

In [20]:
# --- prompt-building utilities ----------------------------------------------
# IMPORTANT: this entire block (helpers + build_prompt) is duplicated VERBATIM
# in app.py. The web UI must produce the exact same prompt strings the model
# saw during training. Any drift here silently degrades inference quality.

def _clean_text(s) -> str:
    """Collapse all whitespace runs into a single space. Keeps a multi-line
    field (e.g. a title with a stray newline) on one line, and is also used
    for the target description."""
    return re.sub(r"\s+", " ", str(s)).strip()

def _or_unspecified(value) -> str:
    """NaN / None / empty string -> 'not specified'. Otherwise the cleaned value."""
    if value is None:
        return "not specified"
    if isinstance(value, float) and pd.isna(value):
        return "not specified"
    s = _clean_text(value)
    return s if s else "not specified"

def build_prompt(row) -> str:
    """Build the model input string from a row of df.

    The exact format below is what the model is trained on; app.py must call
    this identical function so inference-time prompts match training prompts.
    """
    return (
        "generate a job description for the following role.\n"
        f"title: {_or_unspecified(row.get('title'))}\n"
        f"work_type: {_or_unspecified(row.get('formatted_work_type'))}\n"
        f"experience: {_or_unspecified(row.get('formatted_experience_level'))}\n"
        f"location: {_or_unspecified(row.get('location'))}\n"
        f"industry: {_or_unspecified(row.get('industries_joined'))}\n"
        f"skills: {_or_unspecified(row.get('skills_joined'))}"
    )

# Sanity check on the first row.
print(build_prompt(df.iloc[0]))

generate a job description for the following role.
title: Marketing Coordinator
work_type: Full-time
experience: not specified
location: Princeton, NJ
industry: Real Estate
skills: Marketing, Sales


In [21]:
# Subsample. The spec says 200k; if we have fewer rows we just use everything.
N_TARGET = 200_000
n = min(N_TARGET, len(df))
sampled = df.sample(n=n, random_state=SEED).reset_index(drop=True)
print(f"Sampled {len(sampled):,} rows from {len(df):,}")

# Build (prompt, target) pairs for the sampled rows.
print("Building (prompt, target) pairs...")
sampled["prompt"] = sampled.apply(build_prompt, axis=1)
sampled["target"] = sampled["description"].apply(_clean_text)

# 80/10/10 split. Shuffle once with a fixed seed, then slice.
shuffled = sampled.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
n_train = int(0.80 * len(shuffled))
n_val   = int(0.10 * len(shuffled))
train_df = shuffled.iloc[:n_train].reset_index(drop=True)
val_df   = shuffled.iloc[n_train:n_train + n_val].reset_index(drop=True)
test_df  = shuffled.iloc[n_train + n_val:].reset_index(drop=True)
print(f"train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

# Show 3 examples so we can sanity-check the format visually.
for i in range(3):
    print("=" * 80)
    print("PROMPT:")
    print(train_df.iloc[i]["prompt"])
    tgt = train_df.iloc[i]["target"]
    print("\nTARGET (first 300 chars):")
    print(tgt[:300] + ("..." if len(tgt) > 300 else ""))

Sampled 91,971 rows from 91,971
Building (prompt, target) pairs...
train=73,576  val=9,197  test=9,198
PROMPT:
generate a job description for the following role.
title: Medical Technologist/Medical Lab Technician
work_type: Part-time
experience: Mid-Senior level
location: Auburn, WA
industry: Biotechnology Research, Hospitals and Health Care, Research Services
skills: Analyst, Information Technology, Research

TARGET (first 300 chars):
You Belong Here. At MultiCare, we strive to offer a true sense of belonging for all our employees. Across our health care network, you will find a dynamic range of meaningful careers, opportunities for growth, safe workplaces, and flexible schedules. We are connected by our mission - partnering and ...
PROMPT:
generate a job description for the following role.
title: Manufacturing Engineer
work_type: Full-time
experience: Mid-Senior level
location: Germantown, WI
industry: Medical Equipment Manufacturing
skills: Engineering, Information Technology

TARGE

## Section 4 — Tokenize and build a PyTorch Dataset

**What is tokenization?** The model doesn't see strings, it sees integer IDs. The tokenizer splits text into subword pieces (SentencePiece for T5) and looks each piece up in a fixed vocabulary. Output:

- `input_ids` — the integer ID of each token.
- `attention_mask` — `1` for real tokens, `0` for padding (so attention layers ignore the padding).

We pad every example to a fixed length so all items in a batch share a shape; truncation is applied to anything longer than the limit.

**The `-100` label trick.** PyTorch's cross-entropy loss accepts an `ignore_index`, and HuggingFace seq2seq models default to ignoring positions where the label is `-100`. We use this to mask out **padding tokens in the target** — otherwise the model is rewarded for predicting padding, which destroys quality. Concretely: tokenize the target with padding, then replace every pad-token ID with `-100` in the `labels` tensor.

**Lengths.** `max_input_length=256` covers our prompts comfortably (they're short, structured strings). `max_target_length=384` covers most descriptions, with truncation for the long tail.

In [22]:
# Load FLAN-T5-small's tokenizer. AutoTokenizer reads the model's tokenizer
# config from the HuggingFace Hub and returns the matching class
# (T5TokenizerFast, the Rust-backed fast tokenizer). The FIRST call here
# downloads and caches it under ~/.cache/huggingface/.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer: {type(tokenizer).__name__}, vocab size: {tokenizer.vocab_size}")

MAX_INPUT_LEN  = 256
MAX_TARGET_LEN = 384

class JobPostingDataset(Dataset):
    """Tokenize on the fly. We don't pre-tokenize all rows up front because that
    blows up RAM for large datasets; on-the-fly is fast enough thanks to the
    fast Rust tokenizer.

    __getitem__ returns a dict with keys input_ids / attention_mask / labels --
    exactly the keyword args T5ForConditionalGeneration.forward expects. The
    DataLoader's default collate stacks them into a batch of shape (B, L).
    """
    def __init__(self, df, tokenizer, max_input_len=MAX_INPUT_LEN, max_target_len=MAX_TARGET_LEN):
        self.prompts = df["prompt"].tolist()
        self.targets = df["target"].tolist()
        self.tok = tokenizer
        self.max_in  = max_input_len
        self.max_out = max_target_len

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        # Pad to max length so all examples share a shape. (Per-batch padding
        # would save a bit of compute, but per-example is simpler and the perf
        # difference is small.)
        enc_in = self.tok(
            self.prompts[idx],
            max_length=self.max_in,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        enc_out = self.tok(
            self.targets[idx],
            max_length=self.max_out,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        labels = enc_out["input_ids"].squeeze(0)
        # Replace pad-token IDs with -100 so the cross-entropy loss skips them.
        labels = labels.masked_fill(labels == self.tok.pad_token_id, -100)

        return {
            "input_ids":      enc_in["input_ids"].squeeze(0),
            "attention_mask": enc_in["attention_mask"].squeeze(0),
            "labels":         labels,
        }

train_ds = JobPostingDataset(train_df, tokenizer)
val_ds   = JobPostingDataset(val_df,   tokenizer)
test_ds  = JobPostingDataset(test_df,  tokenizer)
print(f"Datasets: train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}")

# Sanity: peek at one tokenized item.
sample = train_ds[0]
print("\nSample keys:", list(sample.keys()))
print("input_ids shape:", sample["input_ids"].shape, "  labels shape:", sample["labels"].shape)
print("First 20 input_ids:", sample["input_ids"][:20].tolist())
print("First 20 labels (-100 means 'ignore in loss'):", sample["labels"][:20].tolist())

Tokenizer: T5Tokenizer, vocab size: 32100
Datasets: train=73,576  val=9,197  test=9,198

Sample keys: ['input_ids', 'attention_mask', 'labels']
input_ids shape: torch.Size([256])   labels shape: torch.Size([384])
First 20 input_ids: [3806, 3, 9, 613, 4210, 21, 8, 826, 1075, 5, 2233, 10, 3721, 7130, 29, 9290, 87, 15789, 138, 8100]
First 20 labels (-100 means 'ignore in loss'): [148, 493, 2961, 947, 5, 486, 4908, 6936, 15, 6, 62, 7131, 12, 462, 3, 9, 1176, 1254, 13, 12770]


In [23]:
BATCH_SIZE = 8

# num_workers=0 on Windows avoids spawn issues inside notebooks. pin_memory speeds
# up the host->GPU copy when training on CUDA (no effect on CPU).
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=(DEVICE.type == "cuda"))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=(DEVICE.type == "cuda"))
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=(DEVICE.type == "cuda"))

# Sanity: pull one batch and confirm shapes.
batch = next(iter(train_loader))
print("Batch shapes:")
for k, v in batch.items():
    print(f"  {k}: {tuple(v.shape)}")

Batch shapes:
  input_ids: (8, 256)
  attention_mask: (8, 256)
  labels: (8, 384)


## Section 5 — Zero-shot baseline

Before training anything, we check what FLAN-T5-small produces with no fine-tuning at all. FLAN-T5 was instruction-tuned by Google, so it can already attempt prompts like *"generate a job description for the following role."* This baseline shows what the model gives us **for free**, so we can later compare it to our fine-tuned model and see exactly what fine-tuning actually adds.

This runs on the pretrained weights freshly downloaded from HuggingFace — nothing has been trained yet. It only takes a minute or two even on CPU.

**Why this matters before committing to training.** Fine-tuning takes hours. If the zero-shot output is already roughly job-description-shaped, fine-tuning will mostly just polish the style. If it's clearly off-task (too short, generic-assistant phrasing, ignoring the input fields), fine-tuning is doing real work — and you have a concrete "before" to point to in the writeup.

**Implementation notes.**

- We define `GEN_KWARGS` here once and reuse it in every later generation step (Section 8 fine-tuned + from-scratch). Changing decoding parameters in one place affects every model.
- We use the **first 3 prompts of `test_df`**, the same ones Section 8 will compare on. That way the four-way comparison in Section 8 is on identical inputs.
- Outputs are cached into a `zeroshot_outputs` list that persists across cells. Section 8 reads it back instead of reloading the pretrained model — saves time and VRAM.
- After printing, we `del model_zeroshot` and clear the CUDA cache so it doesn't compete with the optimizer state during training.

In [24]:
# Generation parameters used by every model below (zero-shot, fine-tuned,
# from-scratch). Defined once here so all three decode with identical settings
# -- changing one knob in this dict affects every model.
GEN_KWARGS = dict(
    max_length=384,
    num_beams=4,
    no_repeat_ngram_size=3,
    early_stopping=True,
)

# Load fresh pretrained FLAN-T5-small. NO fine-tuning has happened yet -- this
# is what the model produces "out of the box" using only Google's instruction-
# tuned weights. We use a separate variable name (model_zeroshot) so that the
# fine-tuning section later can use its own model_pre without any risk of
# overwriting this one.
print(f"Loading pretrained {MODEL_NAME} (zero-shot, no fine-tuning) ...")
model_zeroshot = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE).eval()

@torch.no_grad()
def _zeroshot_generate(prompt: str) -> str:
    enc = tokenizer(prompt, max_length=MAX_INPUT_LEN, truncation=True, return_tensors="pt").to(DEVICE)
    out_ids = model_zeroshot.generate(**enc, **GEN_KWARGS)
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)

# Generate on the SAME first 3 test prompts that the final comparison section
# will use. We cache the outputs in zeroshot_outputs so Section 8 can print them
# back without re-running generation (and without needing to keep the pretrained
# zero-shot model loaded all the way through training).
N_BASELINE = 3
zeroshot_outputs = []  # persists across cells; consumed by the Section 8 comparison

for i in range(min(N_BASELINE, len(test_df))):
    prompt = test_df.iloc[i]["prompt"]
    truth  = test_df.iloc[i]["target"]
    output = _zeroshot_generate(prompt)
    zeroshot_outputs.append(output)

    print("=" * 80)
    print("PROMPT:")
    print(prompt)
    print("\n--- ZERO-SHOT OUTPUT (no fine-tuning):")
    print(output)
    print("\n--- GROUND TRUTH (first 600 chars):")
    print(truth[:600] + ("..." if len(truth) > 600 else ""))
    print()

# Free GPU memory before training -- VRAM is tight on a 1660 Super and we
# don't want this model lingering when the optimizer state shows up.
del model_zeroshot
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print(f"Cached {len(zeroshot_outputs)} zero-shot outputs for the Section 8 comparison.")

Loading pretrained google/flan-t5-small (zero-shot, no fine-tuning) ...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


PROMPT:
generate a job description for the following role.
title: Warehouse Office Coordinator - San Antonio, TX
work_type: Full-time
experience: Entry level
location: San Antonio, TX
industry: Security and Investigations
skills: Administrative

--- ZERO-SHOT OUTPUT (no fine-tuning):
Warehouse Office Coordinator - San Antonio, Texas

--- GROUND TRUTH (first 600 chars):
Convergint is looking for a full-time, enthusiastic, results driven and forward-thinking Warehouse Office Coordinator to join our amazing culture. As a Warehouse Coordinator, you are a part of a dynamic team that allows you to grow as Convergint grows. For information about how we use your personal information, please see our Colleague & Applicant Privacy Notice, available on convergint.com/careers. Who You Are You have a passion for providing world-class service to customers, colleagues and communities. You are a person of integrity with a commitment to growth, accountability and delivering r...

PROMPT:
generate a job 

## Section 6 — Fine-tune FLAN-T5-small

Standard supervised seq2seq recipe:

- Load `T5ForConditionalGeneration.from_pretrained(MODEL_NAME)` — same architecture as Section 4's tokenizer, with weights pretrained by Google on the FLAN mix.
- AdamW, `lr=3e-4`, `weight_decay=0.01`.
- Linear warmup over the first 500 steps, then linear decay to 0 across the remaining steps.
- 3 epochs, gradient clipping at 1.0, gradient accumulation of 2 (so effective batch = 16).
- Mixed-precision (FP16) via `torch.cuda.amp.autocast` + `GradScaler`. **Essential** at 6 GB VRAM — without it the model + optimizer state won't fit.

**HuggingFace concept: `outputs.loss`.** When you call `model(input_ids=..., attention_mask=..., labels=...)`, the model computes the seq2seq cross-entropy loss internally (using `-100`-masked labels we built in Section 4) and returns it as `outputs.loss`. We just call `.backward()` on it — exactly like the loops you wrote from scratch in class.

**Resumability.** Each `save_pretrained(...)` writes a new best-checkpoint to disk along with a `best_val.json` recording the best validation loss. If you re-run this section after a crash, training restarts from epoch 1 but the saved-model file persists and the `best_val` threshold is preserved (so a worse model won't overwrite a better earlier one). For full mid-epoch resume you would need to also save optimizer/scheduler state — this simpler scheme is good enough for a 3-epoch run.

**Time estimate.** 6–10 hours on a 1660 Super at 90k examples. **Many hours longer on CPU** — fix the CUDA install before running this.

In [25]:
import math
import json
import time

# Hyperparameters.
GRAD_ACCUM_STEPS = 2
N_EPOCHS         = 3
LR               = 3e-4
WEIGHT_DECAY     = 0.01
WARMUP_STEPS     = 500
GRAD_CLIP        = 1.0

# How long training will run (in optimizer steps, after gradient accumulation).
steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps     = steps_per_epoch * N_EPOCHS
print(f"steps/epoch={steps_per_epoch}  total_steps={total_steps}")

USE_AMP = (DEVICE.type == "cuda")  # FP16 only makes sense on CUDA. On CPU it's a no-op.

def evaluate(model, loader):
    """Mean validation loss. No grad, eval mode, AMP if on CUDA."""
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for batch in tqdm(loader, desc="val", leave=False):
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                out = model(**batch)
            total += out.loss.item() * batch["input_ids"].size(0)
            count += batch["input_ids"].size(0)
    return total / max(count, 1)

def train_model(checkpoint_dir, model, tag):
    """Manual training loop. Saves to checkpoint_dir whenever val loss improves.
    Resumable: re-running picks up the previous best_val from best_val.json."""
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(exist_ok=True)
    best_path = checkpoint_dir / "best_val.json"
    best_val = float("inf")
    if best_path.exists():
        best_val = json.loads(best_path.read_text())["best_val"]
        print(f"[{tag}] resuming with best_val={best_val:.4f}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps
    )
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    global_step = 0
    for epoch in range(N_EPOCHS):
        model.train()
        running_loss = 0.0
        t0 = time.time()
        pbar = tqdm(train_loader, desc=f"[{tag}] epoch {epoch+1}/{N_EPOCHS}")
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(pbar):
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

            # Forward in mixed precision. T5 returns outputs.loss when labels= is given.
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                out  = model(**batch)
                loss = out.loss / GRAD_ACCUM_STEPS  # scale down for accumulation

            # Backward in scaled FP16.
            scaler.scale(loss).backward()
            running_loss += loss.item() * GRAD_ACCUM_STEPS

            # Step the optimizer every GRAD_ACCUM_STEPS micro-batches.
            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)                                  # bring grads back to FP32 for clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)                                      # actual optimizer step (skipped on inf/nan grads)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

                if global_step % 100 == 0:
                    pbar.set_postfix(
                        loss=f"{running_loss/(step+1):.4f}",
                        lr=f"{scheduler.get_last_lr()[0]:.2e}",
                    )

        avg_train = running_loss / len(train_loader)
        val_loss  = evaluate(model, val_loader)
        dt = time.time() - t0
        print(f"[{tag}] epoch {epoch+1}: train_loss={avg_train:.4f}  val_loss={val_loss:.4f}  ({dt/60:.1f} min)")

        if val_loss < best_val:
            best_val = val_loss
            model.save_pretrained(checkpoint_dir)
            tokenizer.save_pretrained(checkpoint_dir)
            best_path.write_text(json.dumps({"best_val": best_val, "epoch": epoch + 1}))
            print(f"[{tag}]   saved (new best val_loss={best_val:.4f}) -> {checkpoint_dir}")

    return best_val

# Load pretrained FLAN-T5-small. from_pretrained downloads weights the first
# time and caches them under ~/.cache/huggingface/.
print(f"Loading pretrained {MODEL_NAME} ...")
model_pre = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)
print(f"Trainable parameters: {sum(p.numel() for p in model_pre.parameters() if p.requires_grad):,}")

best_pre = train_model(CHECKPOINT_DIR, model_pre, tag="pretrained")
print(f"Pretrained fine-tune done. Best val loss: {best_pre:.4f}")

# Free GPU memory before Section 7.
del model_pre
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

steps/epoch=4599  total_steps=13797
Loading pretrained google/flan-t5-small ...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Trainable parameters: 76,961,152


C:\Users\AlessandroMacchi\AppData\Local\Temp\ipykernel_27432\251659932.py:48: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


[pretrained] epoch 1/3:   0%|          | 0/9197 [00:00<?, ?it/s]

C:\Users\AlessandroMacchi\AppData\Local\Temp\ipykernel_27432\251659932.py:62: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


KeyboardInterrupt: 

## Section 7 — From-scratch comparison (the experiment)

The point of pretraining: a model that already knows grammar, world knowledge, and how to follow instructions adapts to our task with very little data. The **same architecture** initialized **randomly** has to learn everything from scratch — for our ~74k training examples that is far too few. Comparing the two is the experiment for the writeup.

Same data, same optimizer, same epochs, same learning rate. The only difference is `T5ForConditionalGeneration(config)` (random init) instead of `from_pretrained(MODEL_NAME)`.

Expect the from-scratch model to converge slowly and produce repetitive or nonsensical text. That is exactly the result that demonstrates why pretraining matters.

Saved to `./checkpoint_scratch/` (kept separate from `./checkpoint/` so we can load both side-by-side in Section 8 along with the cached zero-shot outputs).

In [ ]:
# Random init, same architecture. T5Config.from_pretrained pulls only the
# architecture spec (number of layers, hidden size, vocab size, etc.) from the
# Hub -- no pretrained weights -- and the bare T5ForConditionalGeneration(config)
# constructor builds the model with PyTorch's default random init.
print("Building from-scratch model with random init (same architecture as FLAN-T5-small)...")
config = T5Config.from_pretrained(MODEL_NAME)
model_scratch = T5ForConditionalGeneration(config).to(DEVICE)
print(f"Trainable parameters: {sum(p.numel() for p in model_scratch.parameters() if p.requires_grad):,}")

# Identical recipe to Section 6: same train_model() function, same hyperparameters,
# same data. Only difference is the starting weights.
best_scratch = train_model(CHECKPOINT_SCRATCH_DIR, model_scratch, tag="scratch")
print(f"From-scratch training done. Best val loss: {best_scratch:.4f}")

# Free GPU memory before Section 8.
del model_scratch
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## Section 8 — Generate samples and compare

Print four model outputs side by side per test prompt:

- the prompt
- the **zero-shot** baseline (cached from Section 5 — no fine-tuning)
- the **fine-tuned** FLAN-T5 output (from `./checkpoint/`)
- the **from-scratch** output (from `./checkpoint_scratch/`)
- the ground-truth description

The zero-shot outputs are read straight from the `zeroshot_outputs` list populated in Section 5, so we don't need to reload the pretrained model again. The fine-tuned and from-scratch checkpoints are loaded fresh from disk.

Same `GEN_KWARGS` as Section 5 (defined there once and reused here), same prompts (`test_df.iloc[:len(zeroshot_outputs)]`). The fine-tuned outputs should look like real job descriptions; the from-scratch ones should look broken or repetitive; the zero-shot output sits in between and tells you exactly what fine-tuning bought.

The same `GEN_KWARGS` are used by `app.py`, so the web UI's outputs match what you see here.

In [ ]:
# GEN_KWARGS was defined in Section 5 -- we deliberately don't redefine it here
# so all three models decode with identical settings.

def load_model(checkpoint_dir):
    return T5ForConditionalGeneration.from_pretrained(checkpoint_dir).to(DEVICE).eval()

print("Loading both fine-tuned checkpoints...")
model_pre_eval     = load_model(CHECKPOINT_DIR)
model_scratch_eval = load_model(CHECKPOINT_SCRATCH_DIR)

@torch.no_grad()
def generate(model, prompt: str) -> str:
    enc = tokenizer(prompt, max_length=MAX_INPUT_LEN, truncation=True, return_tensors="pt").to(DEVICE)
    out_ids = model.generate(**enc, **GEN_KWARGS)
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)

# Compare on exactly the same prompts Section 5 ran zero-shot generation on,
# so the four-way comparison is on identical inputs.
n_show = len(zeroshot_outputs)
for i in range(n_show):
    prompt = test_df.iloc[i]["prompt"]
    truth  = test_df.iloc[i]["target"]
    out_zero = zeroshot_outputs[i]                  # cached from Section 5
    out_pre  = generate(model_pre_eval,     prompt)
    out_sc   = generate(model_scratch_eval, prompt)

    print("=" * 80)
    print("PROMPT:")
    print(prompt)
    print("\n--- ZERO-SHOT OUTPUT (no fine-tuning):")
    print(out_zero)
    print("\n--- FINE-TUNED FLAN-T5 OUTPUT:")
    print(out_pre)
    print("\n--- FROM-SCRATCH OUTPUT:")
    print(out_sc)
    print("\n--- GROUND TRUTH (first 600 chars):")
    print(truth[:600] + ("..." if len(truth) > 600 else ""))
    print()